In [1]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano", model_provider="openai") # 모델 초기화

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("영어로 입력하면 {language}로 해석해줘"),
    HumanMessage("{text}"),
]

prompt = messages.format(language="중국어", text="hi!")
result = model.invoke(prompt)
print(result.content)

AttributeError: 'list' object has no attribute 'format'

In [9]:
model.invoke([{"role": "user", "content": "Hello"}])


AIMessage(content='Hello! How can I assist you today? I can help with explanations, writing, coding, brainstorming, planning, or just chatting—what would you like to do?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 235, 'prompt_tokens': 7, 'total_tokens': 242, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C52Fz7vQqmWcOCN6Xe4pBCFxwHXae', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--76029e24-7cd3-4b19-9112-b00ba29de2db-0', usage_metadata={'input_tokens': 7, 'output_tokens': 235, 'total_tokens': 242, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}})

In [10]:
for token in model.stream(messages):
    print(token.content, end="|")


|안|녕하세요|!||

In [2]:
# Prompt Templates 만드는 코드
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}" 

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [7]:
prompt = prompt_template.invoke({"language": "Italian", "text": "hi!"})

prompt.to_messages()

[SystemMessage(content='Translate the following from English into Italian', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi!', additional_kwargs={}, response_metadata={})]

In [8]:
response = model.invoke(prompt)
print(response.content)

Ciao!


In [10]:
chain = prompt_template | model
response = chain.invoke({"language": "한국어", "text": "hi!"})

response

AIMessage(content='- 안녕! (informal)\n- 안녕하세요! (polite)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 345, 'prompt_tokens': 20, 'total_tokens': 365, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C52o629wff6KegqHOWSbEw8uovqdV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--a589d1a0-7dfd-4e8d-8420-a9913558fae8-0', usage_metadata={'input_tokens': 20, 'output_tokens': 345, 'total_tokens': 365, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 320}})

In [11]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]


In [15]:
from langchain_community.document_loaders import PyPDFLoader

file_path = r"C:\Users\dream\바탕 화면\홍석문_LangChain을 활용한 맞춤형 추천 시스템 설계 및 성능 최적화.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load() #PDF 전체를 읽어서 Document 리스트로 반환

print(len(docs))


3


In [17]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

LangChain을 활용한 맞춤형 추천 시스템 설계 및 성능 최적화 
 
(Design and performance optimization of customized 
recommendation systems with LangChain) 
 
홍석문†                                       하 란†† 
[Seokmun Hong] 

{'producer': 'Microsoft® Word Microsoft 365용', 'creator': 'Microsoft® Word Microsoft 365용', 'creationdate': '2024-12-01T17:15:00+09:00', 'author': '한영진', 'moddate': '2024-12-01T17:17:01+09:00', 'title': '한국정보과학회 학술대회 논문작성양식', 'source': 'C:\\Users\\dream\\바탕 화면\\홍석문_LangChain을 활용한 맞춤형 추천 시스템 설계 및 성능 최적화.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200, add_start_index=True # 한 덩어리 최대 길이, 덩어리끼리 겹치는 부분 길이,  metadata에 원문 내 시작 인덱스를 기록
)

all_splits = text_splitter.split_documents(docs)

len(all_splits)

6

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")#차원 수 : 3072



client=<openai.resources.embeddings.Embeddings object at 0x000002DB5163A990> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002DB5163B250> model='text-embedding-3-large' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


In [ ]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n") 
print(vector_1[:10])


Generated vectors of length 3072

[0.02298710308969021, -0.025805901736021042, -0.016423141583800316, 0.011930267326533794, 0.03096708282828331, -0.02163725718855858, 0.00448294822126627, 0.010249574668705463, -0.02338411659002304, -0.0024317100178450346]


In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = len(embeddings.embed_query("hello world")) #초기화 용
index = faiss.IndexFlatL2(embedding_dim) #차원 수 
#vector_store = FAISS.from_documents(all_splits, documents)
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(), #원본 저장소
    index_to_docstore_id={},
)


In [ ]:
ids = vector_store.add_documents(documents=all_splits)


In [ ]:
results = vector_store.similarity_search(
    "동네 추천 시스템에 어떤 기능이 있어?"
)

print(results[0])

In [ ]:
results = await vector_store.asimilarity_search("동네 추천 시스템에 어떤 기능이 있어?")

print(results[0])

In [ ]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("Langchain이 뭐야?") #문서와 함께 점수도 얻는 예제
doc, score = results[0]
print(f"Score: {score}\n") # 거리 기반이므로 거리 짧을 수록 유사도 높음
print(doc)

In [ ]:
#직접 텍스트를 벡터로 바꿔서 검색하는 방식
embedding = embeddings.embed_query("동네 추천 시스템에 어떤 기능이 있어?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

In [2]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain


@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

NameError: name 'vector_store' is not defined

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

NameError: name 'vector_store' is not defined

LangGraph

In [34]:
from langgraph.prebuilt import create_react_agent

def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_react_agent(
    model="openai:gpt-5-nano",  
    tools=[get_weather],  
    prompt="You are a helpful assistant"  
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='c13d6d62-60ef-4951-afcf-e1bccc5e7f69'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_OPLZPrDhtK7WuOemHMIs1DZO', 'function': {'arguments': '{"city":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 280, 'prompt_tokens': 140, 'total_tokens': 420, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C5NovlzNQ0PwHAlH1gjH39cfGh7jm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--db6d29ec-aa0c-49a3-b214-0688e54591b4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': '

In [ ]:
from langchain.chat_models import init_chat_model #모델 초기화

model = init_chat_model(
    "openai:gpt-5-nano",
    temperature=0 #창의성
)

agent = create_react_agent(
    model=model,
    tools=[get_weather],
)

In [36]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model="openai:gpt-5-nano",
    tools=[get_weather],
    # A static prompt that never changes
    prompt="Never answer questions about the weather."
)

agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='d6e413ad-87f7-4cd4-90b4-226d0990f64c'),
  AIMessage(content='I can’t help with current weather conditions. But I can offer alternatives:\n\n- San Francisco climate at a glance: generally mild year-round with cool summers due to the marine layer, frequent fog in late spring/early summer, and rainfall mainly in the winter (Nov–Apr). Temperatures typically range from the 40s–50s°F (around 5–15°C) in winter to the 60s–70s°F (18–25°C) in summer, with coastal areas cooler and breezeier.\n\n- If you’d like, I can share historical average temperatures by month or by neighborhood, or help you interpret a forecast you already have.\n\n- I can also help you find a reliable source to check current conditions yourself (weather apps, websites, widgets).\n\nTell me what you’d prefer (monthly averages, clothing tips, or a forecast-reading guide), and your preferred units (F or C).', addition

In [39]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver() #현재 프로세스 메모리에 대화 상태 저장

agent = create_react_agent(
    model="openai:gpt-5-nano",
    tools=[get_weather],
    checkpointer=checkpointer,
    prompt= "You are a helpful assistant"  
)

# Run the agent
config = {"configurable": {"thread_id": "1"}}
config2 = {"configurable": {"thread_id": "2"}}
sf_response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]},
    config  
)
ny_response = agent.invoke(
    {"messages": [{"role": "user", "content": "what about new york?"}]},
    config2
)

In [40]:
sf_response
ny_response

{'messages': [HumanMessage(content='what about new york?', additional_kwargs={}, response_metadata={}, id='3513707e-36da-420c-98b2-a3b143bccc58'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_qLdUGcAYuOsuOjvYyfWrTSnI', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 139, 'total_tokens': 291, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C5OAP9muKQCAOMxIuxu7wOGYDEG9A', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--16fee615-086b-487c-980f-28fdb60fd204-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'call_qLdUGcAYuO

In [49]:
from pydantic import BaseModel
from langgraph.prebuilt import create_react_agent

class WeatherResponse(BaseModel):
    conditions: str

agent = create_react_agent(
    model="openai:gpt-5-nano",
    tools=[get_weather],
    response_format=WeatherResponse  
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

result = response["structured_response"].json()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_48860\3985559756.py:17: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  result = response["structured_response"].json()


In [50]:
result["conditions"]
# It's always sunny in San Francisco!


TypeError: string indices must be integers, not 'str'

In [1]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm.invoke("Hello, world!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C6Az8L93J9W69i5v670pCoWUv4CVm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--195cbbe6-db98-4979-a367-4c461e047c69-0', usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})